# Poubelle intelligente : prédiction du niveau de remplissage

**Données simulées** (une mesure toutes les 15 min pendant 30 jours), puis comparaison d'une régression linéaire et d'un Random Forest (MAE et R²).

Objectif : prédire le niveau de remplissage **dans 1 heure** pour déclencher une alerte avant que la poubelle soit pleine.

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

## 2. Génération des données simulées

Le remplissage est plus rapide vers midi et surtout le soir, plus faible la nuit, et plus fort le week-end. Quand le niveau dépasse 90 %, la poubelle est vidée (retour proche de 0 %). Un petit bruit simule l'imprécision du capteur.

In [ ]:
rng = np.random.default_rng(42)

dates = pd.date_range("2026-10-01", periods=30 * 96, freq="15min")  # 96 mesures par jour

def poids_heure(h):
    if h < 7:   return 0.02   # nuit
    if h < 12:  return 0.30   # matin
    if h < 15:  return 0.80   # midi
    if h < 19:  return 0.30   # après-midi
    if h < 22:  return 1.00   # soir
    return 0.20

K = 0.7  # intensité globale du remplissage
niveau = 0.0
niveaux = []
for t in dates:
    moyenne = poids_heure(t.hour) * K
    if t.weekday() >= 5:      # week-end : plus de déchets
        moyenne *= 1.3
    ajout = max(0.0, rng.normal(moyenne, moyenne * 0.5 + 0.05))
    niveau += ajout
    if niveau >= 90:          # la poubelle est vidée
        niveau = rng.uniform(0, 5)
    niveaux.append(niveau)

df = pd.DataFrame({"date": dates, "niveau": niveaux})
df["niveau"] = (df["niveau"] + rng.normal(0, 0.8, len(df))).clip(0, 100)  # bruit capteur
df.to_csv("donnees_poubelle.csv", index=False)
df.head()

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(df["date"], df["niveau"])
plt.axhline(80, color="red", linestyle="--", label="Seuil d'alerte (80 %)")
plt.title("Niveau de remplissage simulé (30 jours)")
plt.ylabel("Niveau (%)")
plt.legend()
plt.show()

## 3. Préparation des variables

Variables : heure, jour de la semaine, niveau actuel, niveau précédent (15 min avant).
Cible : niveau **1 heure plus tard** (4 mesures plus loin).

In [ ]:
HORIZON = 4  # 4 x 15 min = 1 heure

data = pd.DataFrame({
    "heure": df["date"].dt.hour + df["date"].dt.minute / 60,
    "jour_semaine": df["date"].dt.weekday,
    "niveau": df["niveau"],
    "niveau_prec": df["niveau"].shift(1),
    "niveau_futur": df["niveau"].shift(-HORIZON),
}).dropna()

X = data[["heure", "jour_semaine", "niveau", "niveau_prec"]]
y = data["niveau_futur"]

# Séparation dans l'ordre du temps (80 % entraînement, 20 % test)
coupure = int(len(data) * 0.8)
X_train, X_test = X.iloc[:coupure], X.iloc[coupure:]
y_train, y_test = y.iloc[:coupure], y.iloc[coupure:]
print(len(X_train), "lignes d'entraînement,", len(X_test), "lignes de test")

## 4. Entraînement et comparaison

In [ ]:
modeles = {
    "Régression linéaire": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

predictions = {}
resultats = []
for nom, modele in modeles.items():
    modele.fit(X_train, y_train)
    pred = modele.predict(X_test)
    predictions[nom] = pred
    resultats.append({
        "Modèle": nom,
        "MAE": mean_absolute_error(y_test, pred),
        "R²": r2_score(y_test, pred),
    })

pd.DataFrame(resultats).round(3)

## 5. Graphique : prédiction vs réel (3 derniers jours du test)

In [ ]:
n = 3 * 96
plt.figure(figsize=(12, 4))
plt.plot(y_test.values[:n], label="Réel", linewidth=2)
plt.plot(predictions["Régression linéaire"][:n], label="Régression linéaire", alpha=0.7)
plt.plot(predictions["Random Forest"][:n], label="Random Forest", alpha=0.7)
plt.title("Niveau dans 1 heure : réel vs prédit")
plt.ylabel("Niveau (%)")
plt.legend()
plt.show()

## 6. Importance des variables (Random Forest)

In [ ]:
importances = pd.Series(modeles["Random Forest"].feature_importances_, index=X.columns)
importances.sort_values().plot(kind="barh", title="Importance des variables")
plt.show()

## 7. Alerte : niveau prédit supérieur à 80 %

In [ ]:
SEUIL = 80
pred_rf = predictions["Random Forest"]

alerte_predite = pred_rf >= SEUIL
alerte_reelle = y_test.values >= SEUIL

print("Alertes déclenchées :", alerte_predite.sum())
print("Cas réellement pleins dans 1 h :", alerte_reelle.sum())
print("Alertes correctes :", (alerte_predite & alerte_reelle).sum())

## 8. Télécharger le CSV (pour Streamlit)

In [ ]:
try:
    from google.colab import files
    files.download("donnees_poubelle.csv")
except ImportError:
    print("Fichier enregistré : donnees_poubelle.csv")